# H0 Selective-EB 최종 제출 생성

3-seed에서 통과한 고정 구성으로 최종 train 재학습 및 test 추론을 한 번 수행합니다. 출력은 `experiments/gs/notebooks/submission/`에 저장됩니다.

## 고정 계약

- 최종 확률: `0.80 × Selective-EB LR + 0.20 × 자동 LGBM specialist`
- Selective gate: EB LR의 Top-1−Top-2 margin `< 0.05`이면 non-EB LR, 나머지는 EB LR
- margin과 blend 비율은 재탐색하지 않습니다.
- vocabulary, recurrent event, EB 통계, 표준화, specialist 암종쌍은 full train에서만 fit합니다.
- test는 이미 학습한 변환 적용과 예측에만 사용하며 train/test를 결합하지 않습니다.
- 고정 암종명·유전자명·exact mutation 목록은 사용하지 않습니다.

In [1]:
from pathlib import Path
import subprocess, sys
from tqdm.auto import tqdm

ROOT = Path('/Users/admin/Documents/FinalProject/OZ_fianl_hackaton')
EXP_DIR = ROOT / 'experiments' / 'gs' / 'notebooks' / 'exp_model_007'
RUNNER = EXP_DIR / 'common' / 'h0_selective_eb_submission.py'
SUBMISSION_DIR = ROOT / 'experiments' / 'gs' / 'notebooks' / 'submission'
RUN_SUBMISSION = True

assert RUNNER.exists()
assert (ROOT / 'data/raw/train.csv').exists()
assert (ROOT / 'data/raw/test.csv').exists()
assert (ROOT / 'data/raw/sample_submission.csv').exists()
print({'runner': RUNNER, 'output_dir': SUBMISSION_DIR, 'run': RUN_SUBMISSION})

{'runner': PosixPath('/Users/admin/Documents/FinalProject/OZ_fianl_hackaton/experiments/gs/notebooks/exp_model_007/common/h0_selective_eb_submission.py'), 'output_dir': PosixPath('/Users/admin/Documents/FinalProject/OZ_fianl_hackaton/experiments/gs/notebooks/submission'), 'run': True}


/Users/admin/Documents/FinalProject/OZ_fianl_hackaton/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if RUN_SUBMISSION:
    process = subprocess.Popen(
        [sys.executable, str(RUNNER)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    tail = []
    for line in tqdm(process.stdout, desc='final submission training', unit='line'):
        print(line, end='')
        tail = (tail + [line])[-120:]
    if process.wait():
        raise RuntimeError('submission runner failed:\n' + ''.join(tail))
else:
    print('RUN_SUBMISSION=False: 생성하지 않았습니다.')

final submission training: 1line [00:01,  1.60s/line]

[submission] read train, test, and sample submission separately


final submission training: 2line [00:06,  3.58s/line]

[submission] fit train-only structured mutation features


final submission training: 3line [00:47, 20.73s/line]

...

final submission training: 5line [03:17, 55.91s/line]

[submission] fit full-train LGBM and automatic two-pair specialist


final submission training: 6line [04:51, 68.88s/line]

{"submission": "/Users/admin/Documents/FinalProject/OZ_fianl_hackaton/experiments/gs/notebooks/submission/submission_h0_selective_eb_lr_lgbm_specialist_seed42.csv", "audit": "/Users/admin/Documents/FinalProject/OZ_fianl_hackaton/experiments/gs/notebooks/submission/submission_h0_selective_eb_lr_lgbm_specialist_seed42.audit.json", "rows": 2546, "leakage_check": true, "nan_as_mutation_count": 0}


final submission training: 6line [04:51, 48.59s/line]


## 결과 검증

CSV의 ID 순서·행 수·클래스와 audit 계약을 확인합니다.

In [3]:
import json
import pandas as pd

submission_path = SUBMISSION_DIR / 'submission_h0_selective_eb_lr_lgbm_specialist_seed42.csv'
audit_path = submission_path.with_suffix('.audit.json')
submission = pd.read_csv(submission_path)
audit = json.loads(audit_path.read_text(encoding='utf-8'))
sample = pd.read_csv(ROOT / 'data/raw/sample_submission.csv')

assert submission.columns.tolist() == sample.columns.tolist() == ['ID', 'SUBCLASS']
assert submission.ID.equals(sample.ID)
assert len(submission) == len(sample) and submission.SUBCLASS.notna().all()
assert audit['leakage_check'] is True
assert audit['nan_as_mutation_count'] == 0
assert audit['test_read_for_fit_statistics_selection_or_scaling'] is False
display(submission.head())
audit

,ID,SUBCLASS
0,TEST_0000,STES
1,TEST_0001,STES
2,TEST_0002,BRCA
3,TEST_0003,LGG
4,TEST_0004,LUSC


{'run_id': 'submission-h0-selective-eb-lr-lgbm-specialist',
 'model_seed': 42,
 'lr_weight': 0.8,
 'specialist_weight': 0.2,
 'selective_margin': 0.05,
 'threshold_retuned': False,
 'test_role': 'transform_and_predict_only',
 'test_read_for_fit_statistics_selection_or_scaling': False,
 'raw_train_test_concat': False,
 'vocabulary_source': 'full_train_only',
 'specialist_pair_source': 'full_train_only_automatic_discovery',
 'fixed_cancer_gene_exact_mutation_rules': False,
 'nan_as_mutation_count': 0,
 'leakage_check': True,
 'structured_feature_count': 8417,
 'eb_feature_count': 26,
 'final_feature_count': 8443,
 'specialist_pairs': [['KIPAN', 'KIRC'], ['GBMLGG', 'LGG']],
 'selective_non_eb_test_rows': 407,
 'convergence_warning_count': 0,
 'output_file': '/Users/admin/Documents/FinalProject/OZ_fianl_hackaton/experiments/gs/notebooks/submission/submission_h0_selective_eb_lr_lgbm_specialist_seed42.csv',
 'row_count': 2546,
 'runtime_seconds': 289.5993355830433}